# Mint Tracker — запуск в Google Colab

Следит за минтами твоих кошельков и шлёт алерт в Discord-канал, когда несколько из них минтят одну коллекцию.

**Подготовь заранее:**
- URL вебхука канала (Настройки канала → Integrations → Webhooks → New Webhook → Copy URL)
- (опц.) ID роли для пинга (Developer Mode → ПКМ по роли → Copy ID)
- ключи Alchemy (robinhood + eth)
- файл `good_wallets.csv` (колонки `address,type`)

Запускай ячейки по порядку.

> Colab отключается после ~12ч или при простое — для 24/7 лучше VPS/Docker.

## 1. Код и зависимости
Если репозиторий приватный — вставь свой GitHub-токен в `TOKEN` (Settings → Developer settings → Personal access tokens). Если публичный — оставь пустым.

In [ ]:
TOKEN = ""  # GitHub personal access token, если репо приватный
REPO = "c8539-coder/main"
BRANCH = "claude/upbeat-knuth-1r9qh0"

import os
auth = f"{TOKEN}@" if TOKEN else ""
if not os.path.isdir("repo"):
    !git clone https://{auth}github.com/{REPO}.git repo
%cd repo
!git fetch origin && git checkout {BRANCH} && git pull
!pip -q install requests python-dotenv
print("OK")

## 2. Загрузить список кошельков
Нажми кнопку и выбери свой `good_wallets.csv`.

In [ ]:
from google.colab import files
import os, shutil
os.makedirs("scripts/nft_top_wallets/out", exist_ok=True)
up = files.upload()
name = list(up)[0]
shutil.move(name, "scripts/nft_top_wallets/out/good_wallets.csv")
import csv
n = sum(1 for _ in csv.DictReader(open("scripts/nft_top_wallets/out/good_wallets.csv", encoding="utf-8")))
print(f"watchlist сохранён: {n} кошельков")

## 3. Настройки
Заполни webhook, ключи Alchemy и (опц.) ID роли.

In [ ]:
import os
os.environ["DISCORD_WEBHOOK_URL"]     = "ВСТАВЬ_URL_ВЕБХУКА"
os.environ["DISCORD_ROLE_ID"]         = ""  # ID роли для пинга или пусто
os.environ["ALCHEMY_RPC_URL"]         = "https://robinhood-mainnet.g.alchemy.com/v2/ТВОЙ_КЛЮЧ"
os.environ["ALCHEMY_MAINNET_RPC_URL"] = "https://eth-mainnet.g.alchemy.com/v2/ТВОЙ_КЛЮЧ"
os.environ["MINT_ALERT_MIN"] = "5"   # тихий алерт
os.environ["MINT_PING_MIN"]  = "15"  # с этого числа пинг роли
os.environ["MINT_PING_STEP"] = "15"  # повторный пинг каждые +N
print("конфиг задан")

## 4. Запуск
Ячейка работает постоянно и пишет лог. Остановить — квадрат ⏹ слева. Держи вкладку открытой.

In [ ]:
!python -m scripts.mint_bot.main